# PathLens-GNN — delivery suite (one Save Version)

This notebook is the remaining GPU operator. One **Save Version** with Internet + Tesla T4 should:

1. Clone `GIT_REF` from GitHub and install the package.
2. Download BioSNAP, write EDA figures (degrees, split sizes, 3-hop zero-mass).
3. Open the sealed test **once** (`STAGE=final`) and score every delivery method.
4. Rebuild comparison figures (nested PathLens hops stay in the family plot only).
5. Write `val_vs_test_mrr.png` from the paired final run.

Download `/kaggle/working/pathlens-stage-output.zip`. File it with `runs/README.md`.

**Do not retune after this run.** The freeze is a negative result on validation MRR. Test numbers are confirmation, not selection.

Imported PathLens cards are scored from in-repo checkpoints (no retrain). SkipGNN / GCN / GraphSAGE / residual retrain with the same seed, select on validation, then score test. Heuristics are parameter-free.

Set `SUITE = "single"` only if you need to rerun one method.

In [ ]:
SUITE = "delivery"  # delivery | single
METHOD = "three_hop"  # used only when SUITE == "single"
STAGE = "final"  # smoke | train | eval | final
GIT_REF = "research/ranking-loss"
RESUME_ARCHIVE = None
FINAL_TEST_TOKEN = "OPEN_SEALED_TEST_ONCE"
DEVICE = "cuda:0"


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
repository_source = str(REPO / "src")
legacy_source = str(REPO / "legacy2" / "src")
if repository_source not in sys.path:
    sys.path.insert(0, repository_source)
if legacy_source not in sys.path:
    sys.path.insert(0, legacy_source)
print(f"SUITE={SUITE} METHOD={METHOD} STAGE={STAGE} DEVICE={DEVICE}")


## EDA

Context-degree histograms, split sizes, and the fraction of validation positives with 3-hop score exactly 0. This is the zero-mass cluster that inflates optimistic-tie MRR for RA / 3-hop.

In [ ]:
import json
from pathlib import Path

from pathlens.constants import CAMPAIGN_ID
from pathlens.evaluation.eda import hop_reachability, summarize_split, write_eda_figures
from pathlens.runtime.runner import ensure_processed
from pathlens.data.processed import load_processed

print(f"DEVICE={DEVICE}")
try:
    import torch

    print(
        f"torch={torch.__version__} cuda={torch.cuda.is_available()} "
        f"name={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}"
    )
except ImportError:
    print("torch not imported")

processed = ensure_processed(
    Path("data/processed") / CAMPAIGN_ID,
    Path("data/raw/biosnap.tsv"),
    download=True,
    strict_identity=True,
)
split = load_processed(processed, allow_test=False)
summary = summarize_split(split)
reach = hop_reachability(split, device=DEVICE)
eda_dir = Path("runs") / CAMPAIGN_ID / "figures" / "eda"
written = write_eda_figures(split, eda_dir, device=DEVICE)
(eda_dir / "summary.json").write_text(json.dumps({"summary": summary, "reach": reach}, indent=2), encoding="utf-8")
print(json.dumps({"summary": summary, "reach": reach}, indent=2))
for path in written:
    print(path)


## One-shot test scoring

`DELIVERY_METHODS` covers heuristics, GNN baselines, the PathLens family, residual 3-hop, and the blend (which also writes RRF). Selection stays on validation. Test arrays are read only in this cell.

In [ ]:
from pathlens.constants import FINAL_TEST_TOKEN as LOCKED_TOKEN
from pathlens.evaluation.catalog import DELIVERY_METHODS
from pathlens.runtime.runner import run_stage

if STAGE == "final" and FINAL_TEST_TOKEN != LOCKED_TOKEN:
    raise SystemExit("STAGE=final requires FINAL_TEST_TOKEN=OPEN_SEALED_TEST_ONCE")

methods = list(DELIVERY_METHODS) if SUITE == "delivery" else [METHOD]
results = []
for method_id in methods:
    print(f"\n===== {method_id} {STAGE} =====", flush=True)
    result = run_stage(
        method_id,
        STAGE,
        device=DEVICE,
        final_test_token=FINAL_TEST_TOKEN,
    )
    ranking = result["filtered_ranking"]
    hard = result["classification"]["hard"]
    line = (
        f"{method_id}: val_mrr={ranking['mrr']:.4f} "
        f"val_hard_auprc={hard['auprc']:.4f}"
    )
    test = result.get("test")
    if test:
        line += (
            f" test_mrr={test['filtered_ranking']['mrr']:.4f} "
            f"test_hard_auprc={test['classification']['hard']['auprc']:.4f}"
        )
    print(line, flush=True)
    results.append(result)

print(f"scored {len(results)} methods")


## Figures and archive

Comparison plots drop nested PathLens hops. The family ladder is a separate PNG. Delivery extras: bipartite hop cartoon, blend α sweep, epoch-selection curves, degree-tertile MRR, val vs test.

In [ ]:
from pathlib import Path

from pathlens.constants import CAMPAIGN_ID
from pathlens.evaluation.figures import copy_figure_files, write_delivery_figures
from pathlens.evaluation.scoreboard import write_scoreboard_tables
from pathlens.runtime.runner import archive_run_dir

campaign = Path("runs") / CAMPAIGN_ID
figure_dir = campaign / "figures"
written = write_delivery_figures(campaign, figure_dir)
write_scoreboard_tables(campaign, figure_dir)
bundle = Path("/kaggle/working/pathlens-stage-output.zip")
staging = Path("/kaggle/working/delivery-bundle")
if staging.exists():
    import shutil

    shutil.rmtree(staging)
copy_figure_files(written, staging / "figures")
eda_dir = figure_dir / "eda"
if eda_dir.is_dir():
    copy_figure_files(sorted(eda_dir.glob("*")), staging / "figures" / "eda")
for method_id in {row["method"] for row in results}:
    src = campaign / method_id / STAGE / "metrics.json"
    if src.is_file():
        dest = staging / method_id / STAGE
        dest.mkdir(parents=True, exist_ok=True)
        dest.joinpath("metrics.json").write_bytes(src.read_bytes())
archive_run_dir(staging, bundle)
print("models scored:", ", ".join(sorted({row["method"] for row in results})))
print("archive:", bundle)
for path in written:
    print(path)
